In [5]:
%load_ext autoreload
%autoreload 2
import pprint, json, math, os, sys, camelot
# sys.path.append(os.path.abspath(dir_path))
import subprocess

import fitz, pdfplumber, ocrmypdf, pprint
import pandas as pd
import numpy as np
from collections import defaultdict
from app.parse_table import TableParser
from app.utils import Helper

helper = Helper()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
pdf_path = r"C:\Users\rando\Office Projects\mywork-repo\TEXT_PDF.pdf"
pdf_path = r"TEXT_PDF.pdf"

In [ ]:
# ==============================
# 1. EXTRACT TEXT
# ==============================
helper = Helper()
text_data = helper.get_pdf_text(pdf_path)
print("TEXT SAMPLE:", text_data[0][:10])


In [7]:
# ==============================
# 2. GET ALL BLOCKS + IMAGES
# ==============================
helper = Helper()
all_data = helper.get_all_pdf_data(pdf_path)
pprint.pprint(all_data[0])
# print(all_data[0])

{'blocks': [{'bbox': (48.18899917602539,
                      45.759185791015625,
                      278.0625305175781,
                      63.159183502197266),
             'lines': [{'bbox': (48.18899917602539,
                                 45.759185791015625,
                                 278.0625305175781,
                                 63.159183502197266),
                        'dir': (1.0, 0.0),
                        'spans': [{'ascender': 0.9269999861717224,
                                   'bbox': (48.18899917602539,
                                            45.759185791015625,
                                            278.0625305175781,
                                            63.159183502197266),
                                   'char_flags': 16,
                                   'color': -16777216,
                                   'descender': -0.2329999953508377,
                                   'flags': 4,
                                 

In [ ]:
# ==============================
# 3. CLIPPED DATA (EDIT BBOX)
# ==============================
helper = Helper()
bboxes = [
    (0, 0, 300, 400),
    (100, 200, 400, 600)
]

try:
    clipped = helper.get_clipped_data(pdf_path, bboxes)
    print("CLIPPED SAMPLE:", clipped[0])
except Exception as e:
    print("Clipping skipped:", e)


In [ ]:
# ==============================
# 4. DRAW LINES + RECTS
# ==============================
lines = [
    ((50, 50), (300, 50)),
    ((100, 100), (400, 100))
]
rects = [(50, 50, 200, 300)]
pages = [1]

output_draw = pdf_path.replace(".pdf", "_drawn.pdf")
helper.draw_lines_on_pdf( pdf_path,lines,rects,pages,output_draw)



Modified PDF saved to: C:\Users\rando\Office Projects\mywork-repo\TEXT_PDF_drawn.pdf


In [6]:
# ==============================
# 5. LINE BOUNDARIES
# ==============================
helper = Helper()
output_path =helper.draw_boundaries_on_lines(pdf_path)
subprocess.Popen([output_path], shell=True)



<Popen: returncode: None args: ['SAMPLE_line_highlighted.pdf']>

In [5]:
# ==============================
# 6. BLOCK BOUNDARIES
# ==============================
helper = Helper()
output_path = helper.draw_boundaries_on_pdf(pdf_path)
subprocess.Popen([output_path], shell=True)



<Popen: returncode: None args: ['SAMPLE_block_highlighted.pdf']>

In [8]:
# ==============================
# 7. SPAN BOUNDARIES
# ==============================
helper = Helper()
pdf_path = r"TABLE_PDF.pdf"
output_path = helper.draw_span_boundaries(pdf_path)
subprocess.Popen([output_path], shell=True)


<Popen: returncode: None args: ['TABLE_PDF_span_highlighted.pdf']>

In [11]:
# ==============================
# 8. BBOX ONLY TEXT
# ==============================
helper = Helper()
pdf_path = r"EDL.pdf"
output_path = helper.mask_outside_bboxes(pdf_path,[(9.11, 208.22, 295.42, 891.45)])
subprocess.Popen([output_path], shell=True)


<Popen: returncode: None args: ['EDL_bbox_mask.pdf']>

In [ ]:
# ==============================
# 9. SINGLE BBOX DRAW
# ==============================
bbox = (100, 100, 400, 400)
helper = Helper()
helper.draw_bboxes_on_pdf(pdf_path, bbox)



In [ ]:
lines = [
    ((110, 0), (110, 812)),# Vertical line
    ((0, 350), (812, 350)),
    ((570, 0), (570, 812))
]
pages = [12, 14,16]
bboxes = [[0, 120, 180, 812],[180, 85, 360, 812]] #[(0, 85, 180, 812),(180, 85, 360, 812),(0,100,270,812),(0,100,350,812)]
pages = [i for i in range(1,110)]
sample_path = ""
Helper.draw_lines_on_pdf(sample_path, lines, bboxes, pages, dry_path)

In [ ]:
import fitz
import pytesseract
from PIL import Image
import io, re

def get_proper_fund_names(path: str):
    title = {}
    pattern ="((?:LI?i?C|BSE|BANK|SMALL|HEALTH|MNEY|[aA]n\\s*open).*?(?:FUND|Path|ETF|FTF|EOF|FOF|PLAN|SAVER|tax saving scheme|small cap stocks)\\s*(?:FUND\\s*OF\\s*FUND)?)"
    with fitz.open(path) as doc:
        for pgn, page in enumerate(doc):
            clip = fitz.Rect(300, 0, 595, 80)
            pix = page.get_pixmap(clip=clip, dpi=300)
            img = Image.open(io.BytesIO(pix.tobytes()))
            text = pytesseract.image_to_string(img)
            cleaned = re.sub("[^A-Za-z0-9\\s\\.,\\-\\(\\)\\+\\%\\:\\&]+", "", text).strip()
            if matches := re.findall(pattern, cleaned, re.DOTALL):
                title[pgn] = " ".join([_ for _ in matches[0].strip().split(" ") if _])
                print(f"{pgn}:matched {matches[0]}")
            # print(f"[OCR] Page {pgn}: {cleaned}")
            # if cleaned:
            #     title[pgn] = cleaned
    return title
path = r"C:\Users\kaustubh.keny\OneDrive - Cogencis Information Services Ltd\Documents\MUTUAL FUND FACTSHEET FY19-25\2021_changed\LIC Mutual Fund\25_31-Dec-21_FS.pdf"
title = get_proper_fund_names(path)

In [ ]:
import re, fitz

def get_proper_fund_names(path: str, pattern:str):
    title = {} 
    with fitz.open(path) as doc:
        for pgn, page in enumerate(doc):
            text = " ".join(page.get_text("text", clip = (160, 640, 600, 812)).split("\n")) #clip = (0, 0, 210, 155)
            text = re.sub("[^A-Za-z0-9\\s\\.,\\-\\(\\)\\+\\%\\:\\&]+", "", text).strip()
            print(f"{pgn}:-{text}")
            if matches := re.findall(pattern, text, re.DOTALL):
                title[pgn] = " ".join([_ for _ in matches[0].strip().split(" ") if _ ])
                print(pgn,matches[0])
    return title
path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\mywork-repo\35_31-Dec-25_FS.pdf"
pattern = "((?:SBI|i\\s*_|S35).*?(?:Fund\\s*(?:\\-?\\s*Investment\\s*Plan|\\-?\\s*Savings\\s*Plan)?|Index|Saver|ETF|FTF|F[oO]F)\\s*(?:of [Ff]unds?|.*?Aggressive\\s*Plan|.*?Hybrid\\s*Plan|.*?Conservative\\s*Plan)?)"
# pattern = "(MIRAE.*?)NSE\\s*[Ss]ymbol"
title = get_proper_fund_names(path,pattern)

In [ ]:
def extract_clipped_data(input:str, pages:list, bboxes:list):
        
        document = fitz.open(input)
        final_list = []
    
        for pgn in pages:
            page = document[pgn]
            
            all_blocks = [] #store every data from bboxes
            
            for bbox in bboxes:
                blocks, seen_blocks = [], set()  #store unique blocks based on content and bbox
                
                page_blocks = page.get_text('dict', clip=bbox)['blocks']
                for block in page_blocks:
                    if block['type'] == 0 and 'lines' in block: #type 0 means text block
                        #hash_key
                        block_key = (tuple(block['bbox']), tuple(tuple(line['spans'][0]['text'] for line in block['lines'])))
                        if block_key not in seen_blocks:
                            seen_blocks.add(block_key)
                            blocks.append(block)

                sorted_blocks = sorted(blocks, key=lambda x: (x['bbox'][1], x['bbox'][0]))
                all_blocks.append(sorted_blocks)

            final_list.append({
                "pgn": pgn,
                "block": all_blocks #will be list[list,list,..]
            })

        document.close()
        return final_list
    
def extract_data_relative_line(path: str, line_x: float, side: str):
    doc = fitz.open(path)
    pages = doc.page_count

    final_list = []

    for pgn in range(pages):
        page = doc[pgn]

        blocks = page.get_text("dict")["blocks"]
        sorted_blocks = sorted(blocks, key=lambda x: (x["bbox"][1], x["bbox"][0]))
        extracted_blocks = []

        # Keep track of blocks to avoid duplicates
        added_blocks = set()

        for block in sorted_blocks:
            block_id = id(block)  # Unique identifier for the block

            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    origin = span["origin"]
                    x0, _ = origin

                    # Check the side condition
                    if side == "left" and x0 < line_x and block_id not in added_blocks:
                        extracted_blocks.append(block)
                        added_blocks.add(block_id)  # Mark block as added
                    elif side == "right" and x0 > line_x and block_id not in added_blocks:
                        extracted_blocks.append(block)
                        added_blocks.add(block_id)  # Mark block as added

      
        final_list.append({
            "pgn": pgn,
            "blocks": extracted_blocks
        })

    doc.close()

    return final_list
  
def get_clipped_data(input:str, bboxes:list[set], *args):
    
        document = fitz.open(input)
        final_list = []
        if args:
            pages = list(args)
        else:
            pages = [i for i in document.page_count]
        
        for pgn in pages:
            page = document[pgn]

            blocks = []
            for bbox in bboxes:
                blocks.extend(page.get_text('dict', clip = bbox)['blocks']) #get all blocks
            
            filtered_blocks = [block for block in blocks if block['type']== 0 and 'lines' in block]
            # sorted_blocks = sorted(filtered_blocks, key= lambda x: (x['bbox'][1], x['bbox'][0]))
             # Extract text from sorted blocks
            extracted_text = []
            for block in filtered_blocks:
                block_text = []
                for line in block['lines']:
                    line_text = " ".join(span['text'] for span in line['spans'])
                    block_text.append(line_text)
                extracted_text.append("\n".join(block_text))
            
            final_list.append({
            "pgn": pgn,
            "block": filtered_blocks,
            "text": extracted_text
            })
            
            
        document.close()
        return final_list
    
def extract_clipped_text_all_pages(pdf_path, clip_coords):
    results = {}
    doc = fitz.open(pdf_path)
    clip_rect = fitz.Rect(*clip_coords)
    try:
        for page_number, page in enumerate(doc):
            text = page.get_text("text", clip=clip_rect).strip()
            results[page_number] = text
    finally:
        doc.close()
    return results